## Now We will Train the Models on the Data that we Cleaned

In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore') # To ignore all kinds of warnings


In [4]:
#Loading the data from csv
X_train,X_test,y_train,y_test = pd.read_csv('X_train.csv'),pd.read_csv('X_test.csv'),pd.read_csv('y_train.csv'),pd.read_csv('y_test.csv')

In [5]:
#Importing the models and the evaluation metrics
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

In [6]:
def model_eval(model):
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print("Accuracy:", accuracy_score(y_test, y_pred))

In [7]:
models = {"Logistic Regression":LogisticRegression(),"Support Vector Classifier":SVC(),"Naive Bayes":GaussianNB(),"K Nearest Neighbours":KNeighborsClassifier(),"Decision Tree":DecisionTreeClassifier(),
        "Random Forest":RandomForestClassifier(),"AdaBoost":AdaBoostClassifier(),"Gradient Boost":GradientBoostingClassifier(),"XG Boost":XGBClassifier()}
        

In [8]:
#Getting the predictions from all the models
for name,model in models.items():
    print(f"=========== {name} ===========")
    model_eval(model)

=========== Logistic Regression ===========
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1249
           1       0.54      0.81      0.65       512

    accuracy                           0.74      1761
   macro avg       0.72      0.76      0.72      1761
weighted avg       0.79      0.74      0.75      1761

[[894 355]
 [ 99 413]]
Accuracy: 0.7421919363997729
=========== Support Vector Classifier ===========
              precision    recall  f1-score   support

           0       0.87      0.74      0.80      1249
           1       0.54      0.73      0.62       512

    accuracy                           0.74      1761
   macro avg       0.70      0.74      0.71      1761
weighted avg       0.77      0.74      0.75      1761

[[930 319]
 [139 373]]
Accuracy: 0.7399204997160704
=========== Naive Bayes ===========
              precision    recall  f1-score   support

           0       0.91      0.66      0.76      1249
   

### Now we will do Hyperparameter Tuning in order to optimize the performance of the model

#### Logistic Regression, Naive Bayes, and KNN are usually good as baseline models. So we will not be focusing on them

In [9]:
svm_params={"C": [0.1, 1, 10, 100],
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"],
        "degree": [2, 3, 4]
    }

dt_params={
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 3, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": [None, "sqrt", "log2"]
    }

rf_params={
        "n_estimators": [100, 200, 300],
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    }

ada_params= {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 0.5, 1.0]
    }

gb_params={
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }

xg_params= {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "gamma": [0, 0.1, 0.5],
        "reg_alpha": [0, 0.01, 0.1],
        "reg_lambda": [1, 1.5, 2]
    }



In [10]:
#Models whose cross_validation  we will do
cv_models= [("Decision Tree",DecisionTreeClassifier(),dt_params),("Random Forest",RandomForestClassifier(),rf_params),("AdaBoost",AdaBoostClassifier(),ada_params),
            ("Gradient Boost",GradientBoostingClassifier(),gb_params),("XG Boost",XGBClassifier(),xg_params),("Support Vector Classifier",SVC(),svm_params)]

In [ ]:
# We will do it using GridSearchCV and get the best parameters which will be used later
from sklearn.model_selection import GridSearchCV
for name,model,params in cv_models:
    grid = GridSearchCV(estimator=model,
    param_grid=params,
    scoring="f1", # In these kinds of models f1 score matters more than just accuracy
    cv=5,
    n_jobs=-1)
    grid.fit(X_train,y_train)
    print(f"The best parameters for {name} are: {grid.best_params_}")

In [12]:
#Putting the best parameters in the models
tuned_models = {"Support Vector Classifier":SVC(C= 10, degree= 4, gamma= 'scale', kernel= 'poly'),
                "Decision Tree":DecisionTreeClassifier(criterion= 'log_loss', max_depth= 20, max_features= 'sqrt', min_samples_leaf= 1, min_samples_split= 2),
                "Random Forest":RandomForestClassifier(criterion= 'gini', max_depth= 20, max_features= 'sqrt', min_samples_leaf= 1, min_samples_split= 2, n_estimators= 300),"AdaBoost":AdaBoostClassifier(learning_rate= 1.0, n_estimators= 100),
                "Gradient Boost":GradientBoostingClassifier(learning_rate= 0.05, max_depth= 7, min_samples_leaf= 1, min_samples_split= 2, n_estimators= 100, subsample= 0.8),
                "XG Boost":XGBClassifier(colsample_bytree= 1.0, gamma= 0.1, learning_rate= 0.05, max_depth=7, n_estimators=100, reg_alpha= 0.01, reg_lambda= 1.5, subsample= 0.8)}

In [13]:
#Training and testing the  models after tuning
for name,model in tuned_models.items():
    print(f"=========== {name} after Tuning ===========")
    model_eval(model)

=========== Support Vector Classifier after Tuning ===========
              precision    recall  f1-score   support

           0       0.83      0.77      0.80      1249
           1       0.53      0.62      0.57       512

    accuracy                           0.73      1761
   macro avg       0.68      0.70      0.69      1761
weighted avg       0.74      0.73      0.73      1761

[[966 283]
 [196 316]]
Accuracy: 0.7279954571266326
=========== Decision Tree after Tuning ===========
              precision    recall  f1-score   support

           0       0.81      0.79      0.80      1249
           1       0.51      0.55      0.53       512

    accuracy                           0.72      1761
   macro avg       0.66      0.67      0.66      1761
weighted avg       0.72      0.72      0.72      1761

[[982 267]
 [230 282]]
Accuracy: 0.7177739920499716
=========== Random Forest after Tuning ===========
              precision    recall  f1-score   support

           0       0.8

In [15]:
xgb_model = XGBClassifier(colsample_bytree=1.0, gamma=0.1, learning_rate=0.05, max_depth=7, n_estimators=100, reg_alpha=0.01, reg_lambda=1.5, subsample=0.8)
xgb_model.fit(X_train, y_train)

y_prob = xgb_model.predict_proba(X_test)[:, 1]
y_pred_custom = (y_prob >= 0.4).astype(int)
print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.88      0.77      0.82      1249
           1       0.57      0.75      0.65       512

    accuracy                           0.76      1761
   macro avg       0.73      0.76      0.73      1761
weighted avg       0.79      0.76      0.77      1761



#### Looking at all the Recall Scores we can conclude that XGBoost Performs the Best

In [16]:
#Saving the model for later use
import joblib

joblib.dump(xgb_model, 'best_model.pkl')

['best_model.pkl']